# PINNs + Poisson + Last-Layer Laplace

This notebook introduces physics-informed neural networks on Poisson problems.
We use a short 1D warm-up and then a 2D main example with sparse supervised
anchor points plus physics residual training. Uncertainty is quantified with a
last-layer Laplace approximation over the sparse anchors.

Primary references:

- Raissi, Perdikaris, Karniadakis (2019), *Physics-informed neural networks*
- MacKay (1992), *A Practical Bayesian Framework for Backpropagation Networks*
- Ritter, Botev, Barber (2018), *A Scalable Laplace Approximation for Neural Networks*


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / 'src').exists():
    repo_root = repo_root.parent.parent
sys.path.insert(0, str(repo_root / 'src'))

import math
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset

from deepuq.methods import LaplaceWrapper
from deepuq.models import PINN1D, PINN2D

torch.manual_seed(23)
np.random.seed(23)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


## 1D warm-up

We solve

$$
-u''(x)=\pi^2\sin(\pi x),\qquad u(0)=u(1)=0,
$$

whose exact solution is $u(x)=\sin(\pi x)$.


In [ ]:
x_colloc = torch.linspace(0.0, 1.0, 64, requires_grad=True).unsqueeze(-1)
x_bc = torch.tensor([[0.0], [1.0]])
y_bc = torch.zeros(2, 1)
x_anchor = torch.linspace(0.1, 0.9, 8).unsqueeze(-1)
y_anchor = torch.sin(math.pi * x_anchor)

pinn1d = PINN1D(hidden_dims=(64, 64, 64)).to(device)
opt = torch.optim.Adam(pinn1d.parameters(), lr=2e-3)
for _ in range(600):
    opt.zero_grad(set_to_none=True)
    x_c = x_colloc.to(device).requires_grad_(True)
    u = pinn1d(x_c)
    du = torch.autograd.grad(u.sum(), x_c, create_graph=True)[0]
    d2u = torch.autograd.grad(du.sum(), x_c, create_graph=True)[0]
    forcing = (math.pi ** 2) * torch.sin(math.pi * x_c)
    residual = -d2u - forcing
    loss_pde = (residual ** 2).mean()
    loss_bc = torch.nn.functional.mse_loss(pinn1d(x_bc.to(device)), y_bc.to(device))
    loss_anchor = torch.nn.functional.mse_loss(pinn1d(x_anchor.to(device)), y_anchor.to(device))
    loss = loss_pde + 10.0 * loss_bc + 5.0 * loss_anchor
    loss.backward()
    opt.step()

x_eval = torch.linspace(0.0, 1.0, 200).unsqueeze(-1)
y_true = torch.sin(math.pi * x_eval)
with torch.inference_mode():
    y_pred = pinn1d(x_eval.to(device)).cpu()
rmse_1d = torch.mean((y_pred - y_true) ** 2).sqrt().item()
print({'rmse_1d': rmse_1d})


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(x_eval, y_true, label='true')
plt.plot(x_eval, y_pred, label='PINN prediction')
plt.scatter(x_anchor, y_anchor, s=25, label='anchor data')
plt.title('1D PINN warm-up')
plt.legend()
plt.tight_layout()
plt.show()


## 2D main problem

The main case solves

$$
-\Delta u(x,y)=2\pi^2\sin(\pi x)\sin(\pi y),\qquad u=0\;	ext{on the boundary},
$$

with analytic solution $u(x,y)=\sin(\pi x)\sin(\pi y)$. Sparse anchor points
are concentrated near the boundary, so the Laplace fit should report larger
uncertainty in the less directly observed interior.


In [ ]:
def exact_u(coords):
    return torch.sin(math.pi * coords[:, :1]) * torch.sin(math.pi * coords[:, 1:2])


def forcing(coords):
    return 2.0 * math.pi ** 2 * exact_u(coords)

# Boundary and anchor layouts.
side = torch.linspace(0.0, 1.0, 20)
zeros = torch.zeros_like(side)
ones = torch.ones_like(side)
boundary = torch.cat([
    torch.stack([side, zeros], dim=1),
    torch.stack([side, ones], dim=1),
    torch.stack([zeros, side], dim=1),
    torch.stack([ones, side], dim=1),
], dim=0)
anchor = torch.cat([
    torch.rand(40, 2) * 0.25,
    0.75 + torch.rand(40, 2) * 0.25,
], dim=0).clamp(0.0, 1.0)
interior = torch.rand(512, 2)

pinn2d = PINN2D(hidden_dims=(128, 128, 128, 128)).to(device)
opt = torch.optim.Adam(pinn2d.parameters(), lr=2e-3)
for _ in range(1000):
    opt.zero_grad(set_to_none=True)
    xy = interior.to(device).requires_grad_(True)
    u = pinn2d(xy)
    grad = torch.autograd.grad(u.sum(), xy, create_graph=True)[0]
    d2ux = torch.autograd.grad(grad[:, :1].sum(), xy, create_graph=True)[0][:, :1]
    d2uy = torch.autograd.grad(grad[:, 1:].sum(), xy, create_graph=True)[0][:, 1:]
    residual = -(d2ux + d2uy) - forcing(xy)
    loss_pde = (residual ** 2).mean()
    loss_bc = torch.nn.functional.mse_loss(pinn2d(boundary.to(device)), torch.zeros(boundary.size(0), 1, device=device))
    loss_anchor = torch.nn.functional.mse_loss(pinn2d(anchor.to(device)), exact_u(anchor.to(device)))
    loss = loss_pde + 10.0 * loss_bc + 5.0 * loss_anchor
    loss.backward()
    opt.step()

# Fit Laplace only on sparse anchor observations.
anchor_loader = DataLoader(TensorDataset(anchor.float(), exact_u(anchor).float()), batch_size=32, shuffle=True)
la = LaplaceWrapper(pinn2d, likelihood='regression', hessian_structure='diag', subset_of_weights='last_layer')
la.fit(anchor_loader, prior_precision=10.0)


In [ ]:
grid = torch.linspace(0.0, 1.0, 50)
GX, GY = torch.meshgrid(grid, grid, indexing='ij')
coords_eval = torch.stack([GX.reshape(-1), GY.reshape(-1)], dim=1)
true_field = exact_u(coords_eval).reshape(50, 50)
with torch.inference_mode():
    pred_map = pinn2d(coords_eval.to(device)).cpu().reshape(50, 50)
    uq = la.predict_uq(coords_eval.float(), n_samples=40)
mean_field = uq.mean.reshape(50, 50)
std_field = uq.total_var.sqrt().reshape(50, 50)
rmse_2d = torch.mean((pred_map - true_field) ** 2).sqrt().item()
print({'rmse_2d': rmse_2d, 'mean_std': std_field.mean().item()})


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, field, title, cmap in [
    (axes[0], true_field, 'true solution', 'viridis'),
    (axes[1], pred_map, 'PINN mean', 'viridis'),
    (axes[2], torch.abs(pred_map - true_field), 'abs error', 'magma'),
    (axes[3], std_field, 'Laplace std', 'plasma'),
]:
    im = ax.imshow(field, origin='lower', cmap=cmap)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

plt.figure(figsize=(5, 4))
plt.scatter(anchor[:, 0], anchor[:, 1], s=12, label='anchor data')
plt.scatter(boundary[:, 0], boundary[:, 1], s=4, label='boundary points')
plt.title('Sparse supervision layout')
plt.legend(loc='upper center', ncol=2)
plt.tight_layout()
plt.show()
